[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kawaritai/papermage/blob/main/examples/COLAB_concise_demo.ipynb)

# papermage
### "Entity"
Each `Entity` object stores information about its contents and position:

- `.spans: List[Span]` - A `Span` is a pointer into `Document.symbols` (that is, `Span(start=0, end=5)` corresponds to `symbols[0:5]`). By default, when you iterate over an `Entity`, you iterate over its `.spans`.

- `.boxes: List[Box]` - A `Box` represents a rectangular region on the page. Each span is associated a `Box`.

- `.metadata: Metadata` - A free form dictionary-like object to store extra metadata about that `Entity`. These are usually empty.
### "Document"
A `Document` is created by stitching together 3 types of tools: **Parsers**, **Rasterizers** and **Predictors**.

- **Parsers** take a PDF as input and return a `Document` composed of `.symbols` and other layers. The example one we use is a wrapper around PDFPlumber - MIT License utility.

- **Rasterizers** take a PDF as input and return an `Image` per page that is added to `Document.images`. The example one we use is PDF2Image - MIT License.

- **Predictors** take a `Document` and apply some operation to compute a new set of `Entity` objects that we can insert into our `Document`. These are all built in-house and can be either simple heuristics or full machine-learning models.

In [ ]:
# FOR COLAB. IF YOU'RE NOT IN COLAB, GO AWAY
!git clone https://github.com/Kawaritai/papermage.git
%cd papermage
!pip install -r requirements.txt
!pip install torchdata==0.7.1
!apt-get install poppler-utils

In [ ]:
import papermage.recipes as recipes

# DOCUMENT_PATH = "tests/fixtures/papermage.pdf" # from the tests directory
DOCUMENT_PATH = "/path/to/your.pdf"

# Run recipe on our document
doc = recipes.CoreRecipe().run(DOCUMENT_PATH)

In [ ]:
from papermage.magelib import Document

# Go to `papermage.magelib.Document` for documentation
assert type(doc) == Document

print("SPECIAL_FIELDS:",doc.SPECIAL_FIELDS)

display(doc) # shows the same list as doc._layers

In [ ]:
# Document Title
print(f"len(doc.titles): {len(doc.titles)}")

title = doc.titles[0]

print(f"title.spans:", title.spans)
print(f"title.boxes:", title.boxes)
print(f"title.text:", title.text)

In [ ]:
# You can only query Layers using `intersect_by_span` or `intersect_by_box`
print(f"Number of layer kinds: {len(doc.layers)}")
for layer in doc.layers:
    print(f"{layer:15}: {type(doc.__getattribute__(layer))}")

In [ ]:
from papermage.visualizers import plot_entities_on_page
import ipywidgets as widgets
from IPython.display import display, clear_output
import colorsys

def showcase_annotations():
    # Helper to generate a colourmap of n colours
    def get_colors(n):
        return ['#%02x%02x%02x' % tuple(int(c*255) for c in colorsys.hsv_to_rgb(i/n, 0.8, 0.9)) for i in range(n)]

    COLOURS = get_colors(len(doc.layers))

    # Create page slider
    page_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=len(doc.pages) - 1,
        step=1,
        description='Page:',
        continuous_update=False,
        layout=widgets.Layout(width='400px')
    )

    # Create checkboxes for each layer
    checkboxes = {layer: widgets.Checkbox(value=False, description=layer) for layer in doc.layers}
    output = widgets.Output()

    def update_checkbox_labels():
        """Update checkbox labels with entity counts for the current page."""
        page = doc.pages[page_slider.value]
        for layer in doc.layers:
            count = 0
            try:
                # Try to get counts from both within_span and within_box
                span_values = page.within_span(layer)
                box_values = page.within_box(layer)
                # Use the larger count (they might overlap)
                count = max(len(span_values), len(box_values))
            except (TypeError, AttributeError):
                # If we error, count stays 0
                pass

            checkboxes[layer].description = f"{layer} ({count})"

    def update_plot(change):
        # Update checkbox labels when page changes
        if change is None or change.get('name') == 'value' and change['owner'] == page_slider:
            update_checkbox_labels()
        
        with output:
            clear_output(wait=True)
            page = doc.pages[page_slider.value]
            highlighted = page.images[0]
            for i, layer in enumerate(doc.layers):
                if checkboxes[layer].value:  # Only plot if checked
                    try:
                        # within_span
                        values = page.within_span(layer)
                        highlighted = plot_entities_on_page(highlighted, values, box_color=COLOURS[i], box_alpha=0.05)
                        
                        # within_box
                        values = page.within_box(layer)
                        highlighted = plot_entities_on_page(highlighted, values, box_color=COLOURS[i], box_alpha=0.05)
                    except TypeError:
                        pass
            display(highlighted)

    # Attach update function to slider and checkboxes
    page_slider.observe(update_plot, names='value')
    for cb in checkboxes.values():
        cb.observe(update_plot, names='value')

    # Layout: slider on top, controls on left, image on right
    controls = widgets.VBox(list(checkboxes.values()))
    bottom_layout = widgets.HBox([controls, output])
    full_layout = widgets.VBox([page_slider, bottom_layout])

    display(full_layout)
    update_plot(None)  # Initial render

showcase_annotations()